# Flowracle: Physics-Informed Probabilistic Seasonal Streamflow Forecasting Across California's Snow-fed Basins

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/train_flowracle.ipynb)

This notebook executes the causal data alignment, physics-informed feature
engineering, 3-fold expanding-window cross-validation, pre-2021 Bulletin 120
guardrail verification, and production checkpoint training pipeline for
**Flowracle**. Held-out `2021–2025` out-of-sample inference and benchmarking
are separated into the standalone [`evaluate_flowracle.ipynb`](./evaluate_flowracle.ipynb)
notebook so that model training and selection never touch post-2020 data.

### Overview
- **Target**: Monthly Full Natural Flow (FNF, in Acre-Feet) across 25 California
  CDEC snow-fed watersheds (spanning the Sierra Nevada, Southern Cascades, and
  Klamath–Trinity mountains) at 1- to 6-month lead horizons
  ($l \in \{1, \dots, 6\}$), issued on the 1st of each calendar month.
- **Ensemble Architecture** (`flowracle/model.py`): Combines three specialized
  13-quantile XGBoost sub-models—**Anchor** (`models_elev`, 44 features,
  $d_{\max}=5$), **Core** (`models_melt`, 51 features, $d_{\max}=9$), and
  **Anomaly** (`models_resid`, 47 climatology-centered features, $d_{\max}=6$)—
  via out-of-fold monthly inverse-RMSE skill weights, log-space Jensen bias
  correction, and monotonic quantile interpolation to $M = 100$ members.
- **Strict Pre-2021 Temporal Partitioning** (`flowracle/data_pipeline.py`):
  - Fold 1: Train through Dec 2010 $\to$ Validate on 2011–2014
  - Fold 2: Train through Dec 2014 $\to$ Validate on 2015–2017
  - Fold 3: Train through Dec 2018 $\to$ Validate on 2019–2020 (writes the
    frozen production checkpoint `xgb_1634_v135_ensemble.pkl` with zero
    exposure to the 2021–2025 held-out evaluation period).
  *(Each validation slice prepends 6 months starting June 1 of the cutoff year
  solely to populate the 6-month trailing predictor window; those initial 6
  months are never used for target validation.)*

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

## 1. Environment Setup

Set `FLOWRACLE_DATA_DIR` to the directory populated by the preprocessing
notebooks in `notebooks/data_preprocessing/`.

In [ ]:
!if [ ! -d "../flowracle" ] && [ ! -d "google-research/flowracle" ]; then git clone -n --depth=1 --filter=tree:0 https://github.com/google-research/google-research && (cd google-research && git sparse-checkout set --no-cone flowracle && git checkout); fi
!if [ -f "../requirements.txt" ]; then pip install -q -r ../requirements.txt; else pip install -q -r google-research/flowracle/requirements.txt; fi
!pip install -q --no-deps git+https://github.com/neuralgcm/coordax.git
!pip install -q --no-deps git+https://github.com/neuralgcm/terrax.git
!pip install -q chex fiddle jax jax_datetime

In [ ]:
import os
import shutil
import sys

# Run JAX/Terrax data loading on CPU so GPU memory stays dedicated to XGBoost.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import jax

jax.config.update("jax_platform_name", "cpu")

import numpy as np
import pandas as pd

# Ensure the `flowracle` package directory is importable locally or on Colab.
REPO_ROOT = (
    os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(os.getcwd(), "..", "flowracle"))
    else os.path.abspath(
        os.path.join(os.getcwd(), "google-research", "flowracle")
    )
)
if REPO_ROOT not in sys.path:
  sys.path.insert(0, REPO_ROOT)

from flowracle import (
    build_terrax_dataloader,
    compute_composite_cv_objective,
    consolidate_targets,
    evaluate_against_b120,
    fit_and_predict_fn,
    get_batch_predictors_ds,
    get_batch_target_ds,
    get_expanding_window_cv_slices,
    get_issuance_times,
    predict,
)
from flowracle.data_pipeline import DEFAULT_BATCH_SIZE, SYNTHETIC_MONTH_STEP
from flowracle.metrics import compute_fold_validation_ncrps

DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
MODEL_DIR = os.environ.get("FLOWRACLE_OUTPUT_DIR", "./model_output")
os.makedirs(MODEL_DIR, exist_ok=True)
print("FLOWRACLE_DATA_DIR:", DATA_DIR)
print("FLOWRACLE_OUTPUT_DIR:", MODEL_DIR)

## 2. Multi-Source Zarr Ingestion & Causal Terrax DataLoader

`build_terrax_dataloader` aligns all observational sources onto a shared
monthly calendar, subsets SNOTEL stations to the extended California snow-fed
domain (`32.5-42.0°N`, `124.0-115.0°W`), maps irregular calendar month lengths
onto an equicadence 30-day coordinate for Terrax batch serving (recovering true
calendar dates via `real_time` at evaluation time), constructs trailing 5-day
end-of-month daily telemetry windows, and enforces causal lag/lead stencils
(at least a 1-month reporting lag on historical FNF gauge observations).

In [ ]:
aligned = build_terrax_dataloader(data_dir=DATA_DIR)
print(
    "Aligned span (with 6-month target pad): "
    f"{str(aligned.real_time[0])[:10]} to {str(aligned.real_time[-1])[:10]} "
    f"({len(aligned.real_time)} steps)"
)
print(f"FNF watersheds:  {aligned.aux_ds.sizes['station_id']}")
print(f"SNOTEL stations: {aligned.aux_ds.sizes['snotel_index']}")
print(f"CDEC sensors:    {aligned.aux_ds.sizes['ca_sensor_station']}")

train_slices, val_slices = get_expanding_window_cv_slices(aligned.real_time[0])
sample_iter = aligned.dataloader.build_train_inputs(
    aligned.train_inputs_spec,
    batch_size_per_device=DEFAULT_BATCH_SIZE,
    shuffle_buffer_size_in_bytes=50 * 1024 * 1024,
    dataset_rng_seed=42,
    time_sample_offset=SYNTHETIC_MONTH_STEP,
    dataset_time_slice=aligned.real_slice_to_synth(train_slices[0]),
)
sample_batch = next(sample_iter)
for src_name, src_ds in get_batch_predictors_ds(sample_batch).items():
  print(f"  {src_name:18s} -> {dict(src_ds.sizes)}")
print("  targets            ->", dict(get_batch_target_ds(sample_batch).sizes))

## 3. Expanding-Window Cross-Validation & Bulletin 120 Guardrail

Each fold trains from the start of the aligned record up to the fold cutoff,
evaluates probabilistic skill (`nCRPS` combining flow-stratified per-lead CRPS
and spring April–July cumulative CRPS), and computes relative quantile-loss
exceedance against operational California DWR Bulletin 120 forecasts. Fold 3
produces the final frozen checkpoint (`xgb_1634_v135_ensemble.pkl`).

In [ ]:
validation_ncrps = []
b120_fold_exceedances = []

for fold_idx, (train_slice, val_slice) in enumerate(
    zip(train_slices, val_slices)
):
  print(f"\n{'=' * 72}")
  print(
      f"FOLD {fold_idx + 1}/{len(train_slices)}: train={train_slice} |"
      f" val={val_slice}"
  )
  print(f"{'=' * 72}")

  train_iter = aligned.dataloader.build_train_inputs(
      aligned.train_inputs_spec,
      batch_size_per_device=DEFAULT_BATCH_SIZE,
      shuffle_buffer_size_in_bytes=50 * 1024 * 1024,
      dataset_rng_seed=42,
      time_sample_offset=SYNTHETIC_MONTH_STEP,
      dataset_time_slice=aligned.real_slice_to_synth(train_slice),
  )
  eval_list = aligned.dataloader.build_eval_inputs(
      aligned.inputs_spec,
      time_sample_offset=SYNTHETIC_MONTH_STEP,
      dataset_time_slice=aligned.real_slice_to_synth(val_slice),
      batch_size_per_device=DEFAULT_BATCH_SIZE,
  )

  val_y_true = consolidate_targets(
      iter(eval_list), source="target_fnf", field="fnf", axis=0
  )
  eval_times = get_issuance_times(iter(eval_list))
  issue_months = pd.DatetimeIndex(eval_times).month

  # Clear output directory prior to training each fold so Fold 3 leaves the final checkpoint.
  for fname in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(fpath) or os.path.islink(fpath):
      os.unlink(fpath)
    elif os.path.isdir(fpath):
      shutil.rmtree(fpath)

  _ = fit_and_predict_fn(
      train_iter, eval_list, aux_ds=aligned.aux_ds, model_dir=MODEL_DIR
  )

  # Strip ground-truth targets before standalone inference verification.
  for batch_item in eval_list:
    batch_item.pop("target_fnf", None)
  val_y_pred = predict(eval_list, aux_ds=aligned.aux_ds, model_dir=MODEL_DIR)

  fold_metrics = compute_fold_validation_ncrps(
      val_y_pred, val_y_true, issue_months
  )
  validation_ncrps.append(fold_metrics["ncrps"])
  print(
      f"Validation nCRPS={fold_metrics['ncrps']:.6f} "
      f"(per-lead={fold_metrics['per_lead_crps']:,.1f} AF, "
      f"AMJJ-cumul={fold_metrics['amjj_cumul_crps']:,.1f} AF)"
  )

  b120_res = evaluate_against_b120(
      eval_list,
      val_y_pred,
      val_y_true,
      aligned.aux_ds,
      data_dir=DATA_DIR,
      plot=True,
  )
  if b120_res["b120_available"] and b120_res["per_station_per_month"]:
    fold_exc = [
        max(0.0, (s_ours - s_b120) / max(s_b120, 1e-8))
        for md in b120_res["per_station_per_month"]
        for s_ours, s_b120 in zip(md["ours"], md["b120"])
    ]
    b120_fold_exceedances.append(float(np.mean(fold_exc)))

cv_summary = compute_composite_cv_objective(
    validation_ncrps, b120_fold_exceedances
)
print(f"\n{'=' * 72}")
print("EXPANDING-WINDOW CROSS-VALIDATION SUMMARY")
print(
    f"  Mean nCRPS:               {cv_summary['mean_ncrps']:.6f}"
    f" (std={cv_summary['std_ncrps']:.6f})"
)
print(f"  Weighted B120 exceedance: {cv_summary['mean_b120_exceedance']:.6f}")
print(f"  Composite CV objective:   {cv_summary['composite_cv_objective']:.6f}")
print(
    "  Frozen checkpoint:       "
    f" {os.path.join(MODEL_DIR, 'xgb_1634_v135_ensemble.pkl')}"
)
print(f"{'=' * 72}")

## 4. Next Step: Standalone Out-of-Sample Evaluation (2021–2025)

Fold 3 has written the frozen pre-2021 ensemble checkpoint to
`${FLOWRACLE_OUTPUT_DIR}/xgb_1634_v135_ensemble.pkl`. To evaluate this
checkpoint on the strictly held-out `2021–2025` test period against operational
California DWR Bulletin 120 forecasts and water-year hydrograph timing metrics,
run [`evaluate_flowracle.ipynb`](./evaluate_flowracle.ipynb) pointing to the
same `FLOWRACLE_DATA_DIR` and `FLOWRACLE_OUTPUT_DIR`.